# Synchronous (single-urn) model -- 2-side run

Back-end: `structure_FNs_singleside_2026`. Reward structure is data-driven (`make_linear_pairs`), the test phase reports per-pair statistics, and there is a full all-pairs sweep for measuring symbolic-distance and serial-position effects.

In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import multiprocessing as mp
from structure_FNs_singleside_2026 import play_sequence

np.set_printoptions(suppress=True)

In [ ]:
# ---- pair / reward construction (no hard-coding of sequence length) ----
def make_linear_pairs(n):
    """Linear transitive-inference pair sets for stimuli 0..n-1, listed as
    (forward, reverse) in i<j order (matches the original 5-stimulus arrays).
      pairs       : adjacent ordered pairs (distance 1)
      nonadjpairs : non-adjacent pairs touching an extreme (0 or n-1) [trained]
      testpairs   : interior non-adjacent pairs (touch neither extreme) [held out]
    Reward rule: the larger stimulus index is the rewarded position.
    Swap this helper out (or edit the *_reward arrays) for circular,
    reverse-on-nonadjacent, rock-paper-scissors, etc.
    """
    ext = {0, n - 1}
    pairs, nonadj, test = [], [], []
    for i in range(n):
        for j in range(i + 1, n):
            d = j - i
            fwd, rev = [i, j], [j, i]
            if d == 1:
                pairs += [fwd, rev]
            elif (i in ext) or (j in ext):
                nonadj += [fwd, rev]
            else:
                test += [fwd, rev]
    def rew(arr):
        if len(arr) == 0:
            return np.zeros((0, 2), dtype=np.int64)
        return np.asarray([[1, 0] if a > b else [0, 1] for a, b in arr], dtype=np.int64)
    pairs = np.asarray(pairs, dtype=np.int64)
    nonadj = np.asarray(nonadj, dtype=np.int64) if nonadj else np.zeros((0, 2), dtype=np.int64)
    test = np.asarray(test, dtype=np.int64) if test else np.zeros((0, 2), dtype=np.int64)
    return pairs, nonadj, test, rew(pairs), rew(nonadj), rew(test)


# ---- experiment parameters ----
terms = 7                         # sequence length; set to 5, 7, 9, ... freely
sides = 2                         # 1 = one stimulus set (adjacent-only); 2 = two sets

pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward = \
    make_linear_pairs(terms)
allpairs        = np.concatenate((pairs, nonadjpairs, testpairs))
allpairs_reward = np.concatenate((pairs_reward, nonadjpairs_reward, testpairs_reward))

plen = len(pairs)
alen = len(allpairs)

maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8                 # training plays per run (long; reduce to test)
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps*1

evalperpair = 2000                # read-only test plays sampled per pair, per run

print('terms =', terms, ' sides =', sides)
print('#adjacent =', plen, ' #all =', alen, ' #testpairs =', len(testpairs))
print('test pairs:', testpairs.tolist())
print('test distances:', np.abs(testpairs[:,0]-testpairs[:,1]).tolist())

In [ ]:
# use this cell for running the code
start = time.perf_counter()

# nprocs = mp.cpu_count()-1
pool = mp.Pool(processes=3)

sg = SeedSequence()
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]

mpseq_final = pool.starmap(play_sequence, [(
    n, rgs[n], rein1, punish1, rein2, punish2, timesteps, nsteps, sides,
    pairs, testpairs, nonadjpairs, allpairs,
    pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward,
    plen, alen, terms, maxvalue, startstop, noise, annealing, runs,
    inertia, blocklength, evalperpair) for n in range(runs)])

pool.close()
pool.join()

final_sigweights = []
final_cumsuc = []
final_cumsucadd = []
final_testcumsucadd = []
final_recweights = []
final_test_pair_stats = []
final_all_pair_stats = []

for res_idx in range(0, len(mpseq_final)):
    final_sigweights.append(mpseq_final[res_idx][0])
    final_cumsuc.append(mpseq_final[res_idx][1])
    final_cumsucadd.append(mpseq_final[res_idx][2])
    final_testcumsucadd.append(mpseq_final[res_idx][3])
    final_recweights.append(mpseq_final[res_idx][4])
    final_test_pair_stats.append(mpseq_final[res_idx][5])
    final_all_pair_stats.append(mpseq_final[res_idx][6])

final_sigweights = np.asarray(final_sigweights)
final_cumsuc = np.asarray(final_cumsuc)
final_cumsucadd = np.asarray(final_cumsucadd)
final_testcumsucadd = np.asarray(final_testcumsucadd)
final_recweights = np.asarray(final_recweights)
final_test_pair_stats = np.asarray(final_test_pair_stats)   # (runs, n_testpairs, 2)
final_all_pair_stats = np.asarray(final_all_pair_stats)     # (runs, n_allpairs, 2)

print('average cumsuc =')
print(np.average(final_cumsuc)/runs)
print(' ')
print('average final-block training rate =')
print(np.average(final_cumsucadd)/nsteps)
print(' ')
# overall held-out test rate, pooled from the per-pair stats (normalization-free)
test_att = final_test_pair_stats[:, :, 0].sum()
test_suc = final_test_pair_stats[:, :, 1].sum()
print('overall test rate (held-out pairs) =')
print(test_suc / test_att)
print(' ')
finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [ ]:
np.save('singleside_2s_MV10_sigweights', final_sigweights)
np.save('singleside_2s_MV10_recweights', final_recweights)
np.save('singleside_2s_MV10_testcumsucadd', final_testcumsucadd)
np.save('singleside_2s_MV10_test_pair_stats', final_test_pair_stats)
np.save('singleside_2s_MV10_all_pair_stats', final_all_pair_stats)
# save the pair arrays too, so distances can be recomputed later
np.save('singleside_2s_MV10_testpairs', testpairs)
np.save('singleside_2s_MV10_allpairs', allpairs)

## Distance / serial-position analysis

In [ ]:
# ---- distance / serial-position analysis helpers ----
def pair_distance(pairs_arr, terms, circular=False):
    """Symbolic distance of each ordered pair. Set circular=True to use the
    shorter way around a ring (min(d, terms-d))."""
    d = np.abs(pairs_arr[:, 0].astype(int) - pairs_arr[:, 1].astype(int))
    if circular:
        d = np.minimum(d, terms - d)
    return d

def per_run_rate(pair_stats, cols):
    """Per-run accuracy pooled over the given pair columns (nan if no attempts)."""
    att = pair_stats[:, cols, 0].sum(axis=1).astype(float)
    suc = pair_stats[:, cols, 1].sum(axis=1).astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(att > 0, suc / att, np.nan)

def distance_table(pairs_arr, pair_stats, terms, circular=False, title=''):
    """Print mean accuracy grouped by symbolic distance, with SE across runs.
    A rising mean_acc with distance is the symbolic distance effect."""
    d = pair_distance(pairs_arr, terms, circular)
    if title:
        print(title)
    print(f"{'distance':>8} | {'n_pairs':>7} | {'mean_acc':>8} | {'SE':>7} | {'n_runs':>6}")
    print('-' * 50)
    for dist in np.unique(d):
        cols = np.where(d == dist)[0]
        run_acc = per_run_rate(pair_stats, cols)
        valid = run_acc[~np.isnan(run_acc)]
        m = valid.mean() if len(valid) else float('nan')
        se = valid.std(ddof=1) / np.sqrt(len(valid)) if len(valid) > 1 else float('nan')
        print(f'{int(dist):>8} | {len(cols):>7} | {m:>8.4f} | {se:>7.4f} | {len(valid):>6}')
    print(' ')

def per_pair_table(pairs_arr, pair_stats, terms, circular=False, title=''):
    """Print mean accuracy for every individual pair (sorted by distance, then
    by pair). Differences among pairs at the same distance reveal serial
    position effects."""
    d = pair_distance(pairs_arr, terms, circular)
    with np.errstate(invalid='ignore', divide='ignore'):
        rates = np.where(pair_stats[:, :, 0] > 0,
                         pair_stats[:, :, 1] / pair_stats[:, :, 0], np.nan)
    mean_acc = np.nanmean(rates, axis=0)
    n_valid = np.sum(~np.isnan(rates), axis=0)
    order = np.lexsort((np.arange(len(pairs_arr)), d))
    if title:
        print(title)
    print(f"{'pair':>10} | {'dist':>4} | {'mean_acc':>8} | {'n_runs':>6}")
    print('-' * 38)
    for i in order:
        pr = f'{int(pairs_arr[i,0])}-{int(pairs_arr[i,1])}'
        print(f'{pr:>10} | {int(d[i]):>4} | {mean_acc[i]:>8.4f} | {int(n_valid[i]):>6}')
    print(' ')

In [ ]:
# ---- distance effect on the HELD-OUT test pairs ----
# (set circular=True if you trained a circular ordering)
distance_table(testpairs, final_test_pair_stats, terms, circular=False,
               title='Symbolic distance effect -- held-out test pairs')

# ---- distance effect across ALL pairs (full spectrum, incl. distance 1) ----
distance_table(allpairs, final_all_pair_stats, terms, circular=False,
               title='Symbolic distance effect -- all pairs')

# ---- per-pair breakdown across all pairs (for serial position effects) ----
per_pair_table(allpairs, final_all_pair_stats, terms, circular=False,
               title='Per-pair accuracy -- all pairs (serial position)')

In [ ]:
# ---- optional: plot accuracy vs symbolic distance ----
# (skip if matplotlib is unavailable)
import matplotlib.pyplot as plt

def distance_means(pairs_arr, pair_stats, terms, circular=False):
    d = pair_distance(pairs_arr, terms, circular)
    xs, ms, ses = [], [], []
    for dist in np.unique(d):
        cols = np.where(d == dist)[0]
        run_acc = per_run_rate(pair_stats, cols)
        valid = run_acc[~np.isnan(run_acc)]
        xs.append(int(dist)); ms.append(valid.mean())
        ses.append(valid.std(ddof=1)/np.sqrt(len(valid)) if len(valid) > 1 else 0.0)
    return np.array(xs), np.array(ms), np.array(ses)

xt, mt, st = distance_means(testpairs, final_test_pair_stats, terms)
xa, ma, sa = distance_means(allpairs, final_all_pair_stats, terms)

plt.figure(figsize=(6, 4))
plt.errorbar(xa, ma, yerr=sa, marker='o', capsize=3, label='all pairs')
plt.errorbar(xt, mt, yerr=st, marker='s', capsize=3, label='held-out test pairs')
plt.axhline(0.5, ls='--', lw=1, color='grey', label='chance')
plt.xlabel('symbolic distance')
plt.ylabel('mean test accuracy')
plt.title('Symbolic distance effect')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## Duplicate-bin diagnostic

In [ ]:
# how many runs converged to sender strategies in which the same magnitude
# pair (receiver urn) encodes two distinct stimulus pairings (cf. paper fn. 6-7)
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype=numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000]
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1]
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

print('runs with duplicate bins:', runs_dup_bins(runs, final_sigweights, terms))